In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/dataset_features.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

,date,home_team,away_team,resultado,home_score,away_score,tournament,elo_diff,home_forma,away_forma,neutral,h2h_victorias_a,h2h_victorias_b,equipo_a,equipo_b
0,1872-11-30,Scotland,England,draw,0.0,0.0,Friendly,-6.0,1.0,1.0,False,0.0,0.0,England,Scotland
1,1873-03-08,England,Scotland,home_win,4.0,2.0,Friendly,28.0,4.0,1.0,False,0.0,0.0,England,Scotland
2,1874-03-07,Scotland,England,home_win,2.0,1.0,Friendly,-12.0,4.0,4.0,False,1.0,0.0,England,Scotland
3,1875-03-06,England,Scotland,draw,2.0,2.0,Friendly,6.0,5.0,5.0,False,1.0,1.0,England,Scotland
4,1876-03-04,Scotland,England,home_win,3.0,0.0,Friendly,20.0,8.0,5.0,False,1.0,1.0,England,Scotland


In [3]:
features = ['elo_diff', 'home_forma', 'away_forma', 'neutral', 'h2h_victorias_a', 'h2h_victorias_b']

X = df[features]
y = df['resultado']

In [4]:
X['neutral'] = X['neutral'].astype(int)

In [5]:
df_ordenado = df.sort_values('date').reset_index(drop=True)

punto_corte = int(len(df_ordenado) * 0.8)

train = df_ordenado.iloc[:punto_corte]
test = df_ordenado.iloc[punto_corte:]

print(train['date'].max())
print(test['date'].min())

2016-03-24 00:00:00
2016-03-24 00:00:00


In [6]:
fecha_corte = df_ordenado.iloc[punto_corte]['date']
train = df_ordenado[df_ordenado['date'] < fecha_corte]
test = df_ordenado[df_ordenado['date'] >= fecha_corte]

print(train.shape)
print(test.shape)

(39825, 15)
(9970, 15)


In [7]:
df.shape

(49795, 15)

In [8]:
X_train = train[features]
X_train['neutral'] = X_train['neutral'].astype(int)
y_train = train['resultado']

X_test = test[features]
X_test['neutral'] = X_test['neutral'].astype(int)
y_test = test['resultado']

In [9]:
def baseline_tonto(elo_diff, margen=25):
    if elo_diff > margen:
        return 'home_win'
    elif elo_diff < -margen:
        return 'away_win'
    else:
        return 'draw'

predicciones_baseline = X_test['elo_diff'].apply(baseline_tonto)

In [10]:
from sklearn.metrics import accuracy_score

accuracy_baseline = accuracy_score(y_test, predicciones_baseline)
print(accuracy_baseline)

0.4485456369107322


In [11]:
df_reciente = df_ordenado[df_ordenado['date'] >= '2015-01-01']

df_reciente.shape

(11051, 15)

In [12]:
punto_corte = int(len(df_reciente) * 0.8)
fecha_corte = df_reciente.iloc[punto_corte]['date']

train = df_reciente[df_reciente['date'] < fecha_corte]
test = df_reciente[df_reciente['date'] >= fecha_corte]

X_train = train[features]
X_train['neutral'] = X_train['neutral'].astype(int)
y_train = train['resultado']

X_test = test[features]
X_test['neutral'] = X_test['neutral'].astype(int)
y_test = test['resultado']

def baseline_tonto(elo_diff, margen=25):
    if elo_diff > margen:
        return 'home_win'
    elif elo_diff < -margen:
        return 'away_win'
    else:
        return 'draw'

predicciones_baseline = X_test['elo_diff'].apply(baseline_tonto)

accuracy_baseline_reciente = accuracy_score(y_test, predicciones_baseline)
print(accuracy_baseline_reciente)

0.45212285456187895


In [13]:
punto_corte = int(len(df_ordenado) * 0.8)
fecha_corte = df_ordenado.iloc[punto_corte]['date']

train = df_ordenado[df_ordenado['date'] < fecha_corte]
test = df_ordenado[df_ordenado['date'] >= fecha_corte]

X_train = train[features]
X_train['neutral'] = X_train['neutral'].astype(int)
y_train = train['resultado']

X_test = test[features]
X_test['neutral'] = X_test['neutral'].astype(int)
y_test = test['resultado']

In [14]:
X_train.shape 

(39825, 6)

In [15]:
X_train.isna().sum()

elo_diff           16610
home_forma             0
away_forma             0
neutral                0
h2h_victorias_a        0
h2h_victorias_b        0
dtype: int64

In [16]:
filas_validas = X_train['elo_diff'].notna()

X_train = X_train[filas_validas]
y_train = y_train[filas_validas]

In [17]:
filas_validas = X_test['elo_diff'].notna()

X_test = X_test[filas_validas]
y_test = y_test[filas_validas]

In [18]:
print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())

0
0


In [19]:
from sklearn.linear_model import LogisticRegression
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

c:\Users\srpab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train_scaled, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [22]:
from sklearn.metrics import accuracy_score, log_loss

predicciones = modelo.predict(X_test_scaled)
accuracy_modelo = accuracy_score(y_test, predicciones)
print(accuracy_modelo)

0.6711783439490446


In [23]:
from sklearn.metrics import confusion_matrix

matriz = confusion_matrix(y_test, predicciones, labels=['home_win', 'draw', 'away_win'])
print(matriz)

[[2647  131  168]
 [ 814  204  484]
 [ 302  166 1364]]


In [24]:
from sklearn.metrics import log_loss

probabilidades = modelo.predict_proba(X_test_scaled)
logloss_modelo = log_loss(y_test, probabilidades, labels=modelo.classes_)
print(logloss_modelo)

0.7179682340911875


In [25]:
import numpy as np

probabilidades_ingenuas = np.full((len(y_test), 3), 1/3)
logloss_ingenuo = log_loss(y_test, probabilidades_ingenuas, labels=modelo.classes_)
print(logloss_ingenuo)

1.0986122886681098


In [26]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10)
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap

In [27]:
predicciones_rf = rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, predicciones_rf)
print(accuracy_rf)

probabilidades_rf = rf.predict_proba(X_test)
logloss_rf = log_loss(y_test, probabilidades_rf, labels=rf.classes_)
print(logloss_rf)

0.6953821656050956
0.6725181172983157


In [28]:
matriz_rf = confusion_matrix(y_test, predicciones_rf, labels=['home_win', 'draw', 'away_win'])
print(matriz_rf)

[[2686  105  155]
 [ 739  372  391]
 [ 357  166 1309]]


In [29]:
df_nuevo = pd.read_csv('../data/raw/results_new.csv')
df_nuevo['date'] = pd.to_datetime(df_nuevo['date'])
print(df_nuevo['date'].max())

2026-06-27 00:00:00


In [30]:
mundial_2026 = df_nuevo[
    (df_nuevo['tournament'] == 'FIFA World Cup') &
    (df_nuevo['date'] >= '2026-06-01')
]

mundial_2026.shape

(72, 9)

In [31]:
mundial_2026.tail(60)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49417,2026-06-15,Belgium,Egypt,1.0,1.0,FIFA World Cup,Seattle,United States,True
49418,2026-06-15,Iran,New Zealand,2.0,2.0,FIFA World Cup,Inglewood,United States,True
49419,2026-06-15,Spain,Cape Verde,0.0,0.0,FIFA World Cup,Atlanta,United States,True
49420,2026-06-15,Saudi Arabia,Uruguay,1.0,1.0,FIFA World Cup,Miami Gardens,United States,True
49421,2026-06-16,France,Senegal,3.0,1.0,FIFA World Cup,East Rutherford,United States,True
49422,2026-06-16,Iraq,Norway,1.0,4.0,FIFA World Cup,Foxborough,United States,True
49423,2026-06-16,Argentina,Algeria,3.0,0.0,FIFA World Cup,Kansas City,United States,True
49424,2026-06-16,Austria,Jordan,3.0,1.0,FIFA World Cup,Santa Clara,United States,True
49425,2026-06-17,Portugal,DR Congo,1.0,1.0,FIFA World Cup,Houston,United States,True
49426,2026-06-17,Uzbekistan,Colombia,1.0,3.0,FIFA World Cup,Mexico City,Mexico,True


In [32]:
mundial_jugados = mundial_2026[mundial_2026['home_score'].notna()]
mundial_jugados.shape

(48, 9)

In [33]:
import pandas as pd

df_actualizado = pd.read_csv('../data/processed/dataset_features.csv')
df_actualizado['date'] = pd.to_datetime(df_actualizado['date'])

mundial_jugados = df_actualizado[
    (df_actualizado['tournament'] == 'FIFA World Cup') &
    (df_actualizado['date'] >= '2026-06-01') &
    (df_actualizado['home_score'].notna())
]

mundial_jugados.shape

(48, 15)

In [34]:
X_mundial = mundial_jugados[features]
X_mundial['neutral'] = X_mundial['neutral'].astype(int)

In [35]:
X_mundial.isna().sum()

elo_diff           18
home_forma          0
away_forma          0
neutral             0
h2h_victorias_a     0
h2h_victorias_b     0
dtype: int64

In [36]:
mundial_jugados[mundial_jugados['elo_diff'].isna()][['home_team', 'away_team']]

,home_team,away_team
49747,Mexico,South Africa
49748,South Korea,Czech Republic
49749,Canada,Bosnia and Herzegovina
49750,United States,Paraguay
49758,Ivory Coast,Ecuador
49760,Iran,New Zealand
49761,Spain,Cape Verde
49762,Saudi Arabia,Uruguay
49770,Portugal,DR Congo
49771,Czech Republic,South Africa


In [37]:
df_elo_wc = pd.read_csv('../data/raw/elo_ratings_wc2026.csv')
df_elo_wc.head()

,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
0,1901,1901-12-31,England,1,EN,2013,1,2079,2,1989,...,38,35,0,46,16,11,262,102,UEFA,0
1,1901,1901-12-31,Scotland,2,SQ,1973,1,2104,1,2018,...,37,37,0,53,9,12,277,101,UEFA,0
2,1902,1902-12-31,Argentina,1,AR,2021,1,2021,1,2021,...,0,1,0,1,0,0,6,0,CONMEBOL,0
3,1902,1902-12-31,England,2,EN,1995,1,2079,2,1989,...,39,38,0,47,16,14,266,105,UEFA,0
4,1902,1902-12-31,Scotland,3,SQ,1983,1,2104,1,2017,...,39,40,0,56,9,14,293,106,UEFA,0


In [38]:
elo_actual = (
    df_elo_wc.sort_values('year')
    .groupby('country')
    .tail(1)[['country', 'rating']]
)

elo_actual.head()

,country,rating
4635,Spain,2165
4636,Argentina,2113
4637,France,2081
4638,England,2020
4639,Brazil,1984


In [39]:
elo_actual[elo_actual['country'].isin(['Cape Verde', 'Bosnia and Herzegovina', 'DR Congo', 'New Zealand'])]

,country,rating
4671,DR Congo,1655
4674,Bosnia and Herzegovina,1594
4675,New Zealand,1585
4677,Cape Verde,1549


In [40]:
filas_con_nan = X_mundial['elo_diff'].isna()

for idx in X_mundial[filas_con_nan].index:
    home = mundial_jugados.loc[idx, 'home_team']
    away = mundial_jugados.loc[idx, 'away_team']
    
    elo_home = elo_actual[elo_actual['country'] == home]['rating'].values
    elo_away = elo_actual[elo_actual['country'] == away]['rating'].values
    
    if len(elo_home) > 0 and len(elo_away) > 0:
        X_mundial.loc[idx, 'elo_diff'] = elo_home[0] - elo_away[0]

In [41]:
X_mundial['elo_diff'].isna().sum()

np.int64(2)

In [42]:
mundial_jugados[X_mundial['elo_diff'].isna()][['home_team', 'away_team']]

,home_team,away_team
49748,South Korea,Czech Republic
49771,Czech Republic,South Africa


In [43]:
elo_actual[elo_actual['country'].str.contains('Czech', case=False, na=False)]

,country,rating
4665,Czechia,1726


In [44]:
rating_czechia = 1726
rating_south_korea = elo_actual[elo_actual['country'] == 'South Korea']['rating'].values[0]
rating_south_africa = elo_actual[elo_actual['country'] == 'South Africa']['rating'].values[0]

X_mundial.loc[49748, 'elo_diff'] = rating_south_korea - rating_czechia
X_mundial.loc[49771, 'elo_diff'] = rating_czechia - rating_south_africa

In [45]:
X_mundial['elo_diff'].isna().sum()

np.int64(0)

In [46]:
X_mundial_scaled = scaler.transform(X_mundial)
pred_logistica = modelo.predict(X_mundial_scaled)

pred_rf = rf.predict(X_mundial)

In [47]:
resumen = mundial_jugados[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'resultado']].copy()
resumen['pred_logistica'] = pred_logistica
resumen['pred_rf'] = pred_rf
resumen['acierto_logistica'] = resumen['resultado'] == resumen['pred_logistica']
resumen['acierto_rf'] = resumen['resultado'] == resumen['pred_rf']

resumen

,date,home_team,away_team,home_score,away_score,resultado,pred_logistica,pred_rf,acierto_logistica,acierto_rf
49747,2026-06-11,Mexico,South Africa,2.0,0.0,home_win,home_win,home_win,True,True
49748,2026-06-11,South Korea,Czech Republic,2.0,1.0,home_win,home_win,home_win,True,True
49749,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,draw,home_win,home_win,False,False
49750,2026-06-12,United States,Paraguay,4.0,1.0,home_win,away_win,away_win,False,False
49751,2026-06-13,Qatar,Switzerland,1.0,1.0,draw,away_win,away_win,False,False
49752,2026-06-13,Brazil,Morocco,1.0,1.0,draw,home_win,draw,False,True
49753,2026-06-13,Haiti,Scotland,0.0,1.0,away_win,away_win,away_win,True,True
49754,2026-06-13,Australia,Turkey,2.0,0.0,home_win,away_win,away_win,False,False
49755,2026-06-14,Sweden,Tunisia,5.0,1.0,home_win,home_win,home_win,True,True
49756,2026-06-14,Netherlands,Japan,2.0,2.0,draw,away_win,away_win,False,False


In [48]:
accuracy_score(resumen['resultado'], resumen['pred_logistica'])


0.6666666666666666

In [49]:
accuracy_score(resumen['resultado'], resumen['pred_rf'])

0.6666666666666666

In [50]:
resumen.to_csv('../outputs/predicciones_fase_grupos_2026-06-27.csv', index=False)

In [51]:
dieciseisavos = pd.read_csv('../data/raw/dieciseisavos_2026.csv')
dieciseisavos['date'] = pd.to_datetime(dieciseisavos['date'])
dieciseisavos

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2026-06-28,South Africa,Canada,NaN,NaN,FIFA World Cup,TBD,TBD,False
1,2026-06-29,Germany,Paraguay,NaN,NaN,FIFA World Cup,TBD,TBD,True
2,2026-06-29,Netherlands,Morocco,NaN,NaN,FIFA World Cup,TBD,TBD,True
3,2026-06-29,Brazil,Japan,NaN,NaN,FIFA World Cup,TBD,TBD,True
4,2026-06-30,France,Sweden,NaN,NaN,FIFA World Cup,TBD,TBD,True
5,2026-06-30,Ivory Coast,Norway,NaN,NaN,FIFA World Cup,TBD,TBD,True
6,2026-06-30,Mexico,Ecuador,NaN,NaN,FIFA World Cup,TBD,TBD,False
7,2026-07-01,United States,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,TBD,TBD,False
8,2026-07-01,Belgium,Senegal,NaN,NaN,FIFA World Cup,TBD,TBD,True
9,2026-07-01,England,DR Congo,NaN,NaN,FIFA World Cup,TBD,TBD,True


In [52]:
dieciseisavos = pd.read_csv('../data/processed/dieciseisavos_features.csv')
dieciseisavos['date'] = pd.to_datetime(dieciseisavos['date'])

X_dieciseisavos = dieciseisavos[features]
X_dieciseisavos['neutral'] = X_dieciseisavos['neutral'].astype(int)

X_dieciseisavos_scaled = scaler.transform(X_dieciseisavos)
pred_logistica_16 = modelo.predict(X_dieciseisavos_scaled)
pred_rf_16 = rf.predict(X_dieciseisavos)

In [53]:
resumen_16 = dieciseisavos[['date', 'home_team', 'away_team']].copy()
resumen_16['pred_logistica'] = pred_logistica_16
resumen_16['pred_rf'] = pred_rf_16

resumen_16

,date,home_team,away_team,pred_logistica,pred_rf
0,2026-06-28,South Africa,Canada,away_win,away_win
1,2026-06-29,Germany,Paraguay,home_win,home_win
2,2026-06-29,Netherlands,Morocco,away_win,draw
3,2026-06-29,Brazil,Japan,home_win,draw
4,2026-06-30,Mexico,Ecuador,home_win,home_win
5,2026-06-30,France,Sweden,home_win,home_win
6,2026-06-30,Ivory Coast,Norway,away_win,away_win
7,2026-07-01,United States,Bosnia and Herzegovina,home_win,home_win
8,2026-07-01,Belgium,Senegal,home_win,home_win
9,2026-07-01,England,DR Congo,home_win,home_win


In [54]:
resumen_16.to_csv('../outputs/predicciones_dieciseisavos_2026-06-28.csv', index=False)

In [55]:
resultados_reales = pd.read_csv('../data/raw/resultados_dieciseisavos.csv')
predicciones = pd.read_csv('../outputs/predicciones_dieciseisavos_2026-06-28.csv')

resumen_16 = predicciones.copy()
resumen_16['resultado_real'] = resultados_reales['resultado']
resumen_16['acierto_logistica'] = resumen_16['pred_logistica'] == resumen_16['resultado_real']
resumen_16['acierto_rf'] = resumen_16['pred_rf'] == resumen_16['resultado_real']

resumen_16

,date,home_team,away_team,pred_logistica,pred_rf,resultado_real,acierto_logistica,acierto_rf
0,2026-06-28,South Africa,Canada,away_win,away_win,away_win,True,True
1,2026-06-29,Germany,Paraguay,home_win,home_win,draw,False,False
2,2026-06-29,Netherlands,Morocco,away_win,draw,draw,False,True
3,2026-06-29,Brazil,Japan,home_win,draw,home_win,True,False
4,2026-06-30,Mexico,Ecuador,home_win,home_win,home_win,True,True
5,2026-06-30,France,Sweden,home_win,home_win,away_win,False,False
6,2026-06-30,Ivory Coast,Norway,away_win,away_win,home_win,False,False
7,2026-07-01,United States,Bosnia and Herzegovina,home_win,home_win,home_win,True,True
8,2026-07-01,Belgium,Senegal,home_win,home_win,home_win,True,True
9,2026-07-01,England,DR Congo,home_win,home_win,home_win,True,True


In [56]:
print(resumen_16['acierto_logistica'].sum(), '/', len(resumen_16))
print(resumen_16['acierto_rf'].sum(), '/', len(resumen_16))

10 / 16
9 / 16
